In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/DSN_Hackathon'   # <-- EDIT to your folder
OUTPUT_DIR = DATA_DIR

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb; HAS_LGB = True
except ImportError: HAS_LGB = False
try:
    import xgboost as xgb; HAS_XGB = True
except ImportError: HAS_XGB = False
try:
    from catboost import CatBoostRegressor; HAS_CAT = True
except ImportError: HAS_CAT = False

print("LightGBM:", HAS_LGB, "| XGBoost:", HAS_XGB, "| CatBoost:", HAS_CAT)

RNG = 42
N_FOLDS = 5
SEEDS = [42, 202, 777]   # bagging over multiple seeds cuts variance

LightGBM: True | XGBoost: True | CatBoost: False


In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

train['is_train'] = 1
test['is_train'] = 0
test['total_sales'] = np.nan
full = pd.concat([train, test], ignore_index=True, sort=False)
print(train.shape, test.shape)

(6818, 14) (1705, 14)


In [ ]:
full['product_category'] = full['product_category'].str.strip().str.lower()
full['fat_content'] = full['fat_content'].str.strip().str.title()

cat_to_broad = {
    'health and hygiene': 'Non_Consumable', 'household': 'Non_Consumable', 'others': 'Non_Consumable',
    'soft drinks': 'Drinks', 'hard drinks': 'Drinks',
}
full['item_type_broad'] = full['product_category'].map(cat_to_broad).fillna('Food')
full.loc[full['item_type_broad'] == 'Non_Consumable', 'fat_content'] = 'Non_Edible'

In [ ]:
w_by_code = full.groupby('product_code')['product_weight_kg'].transform('mean')
full['product_weight_kg'] = full['product_weight_kg'].fillna(w_by_code)
w_by_cat = full.groupby('product_category')['product_weight_kg'].transform('mean')
full['product_weight_kg'] = full['product_weight_kg'].fillna(w_by_cat)
full['product_weight_kg'] = full['product_weight_kg'].fillna(full['product_weight_kg'].mean())

full['store_size'] = full['store_size'].fillna('Unknown')   # structural, whole-store missing

full['shelf_visibility_raw'] = full['shelf_visibility']
full.loc[full['shelf_visibility'] == 0, 'shelf_visibility'] = np.nan
vis_by_code = full.groupby('product_code')['shelf_visibility'].transform('mean')
full['shelf_visibility'] = full['shelf_visibility'].fillna(vis_by_code)
vis_by_cat = full.groupby('product_category')['shelf_visibility'].transform('mean')
full['shelf_visibility'] = full['shelf_visibility'].fillna(vis_by_cat)

In [ ]:
full['price_per_kg'] = full['product_price'] / full['product_weight_kg']
full['log_price'] = np.log1p(full['product_price'])
full['log_weight'] = np.log1p(full['product_weight_kg'])
full['visibility_mean_ratio'] = full['shelf_visibility'] / full.groupby('product_category')['shelf_visibility'].transform('mean')
full['visibility_rank_in_store'] = full.groupby('store_code')['shelf_visibility'].rank(pct=True)
full['price_rank_in_category'] = full.groupby('product_category')['product_price'].rank(pct=True)
full['price_vs_category_median'] = full['product_price'] / full.groupby('product_category')['product_price'].transform('median')

full['store_age_bucket'] = pd.cut(full['store_age_years'], bins=[0, 20, 30, 40, 100],
                                   labels=['New', 'Mid', 'Mature', 'Old'])
full['was_zero_visibility'] = (full['shelf_visibility_raw'] == 0).astype(int)
full['price_bucket'] = pd.qcut(full['product_price'], 10, labels=False, duplicates='drop')

# safe popularity/assortment signals (no target used)
full['product_store_count'] = full.groupby('product_code')['product_code'].transform('count')
full['store_assortment_size'] = full.groupby('store_code')['product_category'].transform('nunique')
full['category_size'] = full.groupby('product_category')['product_category'].transform('count')

size_rank = {'Unknown': -1, 'Small': 0, 'Medium': 1, 'Large': 2}
full['store_size_rank'] = full['store_size'].map(size_rank)
tier_rank = {'Tier_1': 1, 'Tier_2': 2, 'Tier_3': 3}
full['store_location_tier_rank'] = full['store_location_tier'].map(tier_rank)
full['size_x_tier'] = full['store_size_rank'] * full['store_location_tier_rank']

cat_cols = ['product_code', 'fat_content', 'product_category', 'store_code',
            'store_size', 'store_location_tier', 'store_format', 'item_type_broad',
            'store_age_bucket']
for c in cat_cols:
    full[c] = full[c].astype(str)

full['store_x_category'] = full['store_code'] + '_' + full['product_category']
full['format_x_category'] = full['store_format'] + '_' + full['product_category']
full['tier_x_category'] = full['store_location_tier'] + '_' + full['product_category']

In [ ]:
train_fe = full[full['is_train'] == 1].reset_index(drop=True)
test_fe = full[full['is_train'] == 0].reset_index(drop=True)

y = np.log1p(train_fe['total_sales'].values)

TE_COLS = ['product_code', 'product_category', 'store_code', 'store_format',
           'store_location_tier', 'store_size', 'item_type_broad',
           'store_x_category', 'format_x_category', 'tier_x_category']

NUM_COLS = ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years',
            'price_per_kg', 'log_price', 'log_weight', 'visibility_mean_ratio',
            'visibility_rank_in_store', 'price_rank_in_category', 'price_vs_category_median',
            'was_zero_visibility', 'store_size_rank', 'store_location_tier_rank',
            'size_x_tier', 'price_bucket', 'product_store_count',
            'store_assortment_size', 'category_size']

GLOBAL_MEAN = y.mean()

def kfold_target_encode(train_df, test_df, col, y, n_folds=N_FOLDS, smoothing=15):
    oof = np.zeros(len(train_df))
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RNG)
    for tr_idx, val_idx in kf.split(train_df):
        stats = pd.DataFrame({col: train_df[col].values[tr_idx], 'y': y[tr_idx]}).groupby(col)['y'].agg(['mean', 'count'])
        smooth = (stats['mean'] * stats['count'] + GLOBAL_MEAN * smoothing) / (stats['count'] + smoothing)
        oof[val_idx] = pd.Series(train_df[col].values[val_idx]).map(smooth).fillna(GLOBAL_MEAN).values
    stats_full = pd.DataFrame({col: train_df[col].values, 'y': y}).groupby(col)['y'].agg(['mean', 'count'])
    smooth_full = (stats_full['mean'] * stats_full['count'] + GLOBAL_MEAN * smoothing) / (stats_full['count'] + smoothing)
    test_enc = test_df[col].map(smooth_full).fillna(GLOBAL_MEAN).values
    return oof, test_enc

for col in TE_COLS:
    oof_a, test_a = kfold_target_encode(train_fe, test_fe, col, y, smoothing=10)
    oof_b, test_b = kfold_target_encode(train_fe, test_fe, col, y, smoothing=50)
    train_fe[f'te_{col}'] = (oof_a + oof_b) / 2
    test_fe[f'te_{col}'] = (test_a + test_b) / 2

TE_FEATURES = [f'te_{c}' for c in TE_COLS]
FEATURES = NUM_COLS + TE_FEATURES

X = train_fe[FEATURES].values
X_test = test_fe[FEATURES].values
print("Feature matrix:", X.shape, "| Test matrix:", X_test.shape)

Feature matrix: (6818, 29) | Test matrix: (1705, 29)


In [ ]:
!pip install -q optuna

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_FOLDS = 4          # a bit lighter than final N_FOLDS=5, for search speed
N_TRIALS = 50            # raise to 80-100 if you have time budget on Colab
TUNE_SEED = 42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 12.1 MB/s eta 0:00:00


In [ ]:
def cv_rmse_for_params(fit_fn, params):
    """Runs TUNE_FOLDS-fold CV with given params, returns mean RMSE on original scale."""
    kf = KFold(n_splits=TUNE_FOLDS, shuffle=True, random_state=TUNE_SEED)
    fold_scores = []
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        pred_val = fit_fn(params, X_tr, y_tr, X_val, y_val)
        rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(pred_val)))
        fold_scores.append(rmse)
    return float(np.mean(fold_scores)), float(np.std(fold_scores))

In [ ]:
def lgb_fit(params, X_tr, y_tr, X_val, y_val):
    model = lgb.LGBMRegressor(
        n_estimators=5000, random_state=TUNE_SEED, n_jobs=-1, verbose=-1, **params
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(150, verbose=False)])
    return model.predict(X_val)

def lgb_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    mean_rmse, std_rmse = cv_rmse_for_params(lgb_fit, params)
    trial.set_user_attr('std', std_rmse)
    return mean_rmse

if HAS_LGB:
    lgb_study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
    lgb_study.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
    print("Best LightGBM CV RMSE:", lgb_study.best_value)
    print("Best LightGBM params:", lgb_study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

Best LightGBM CV RMSE: 1114.0853509357453
Best LightGBM params: {'learning_rate': 0.04232739958822589, 'num_leaves': 66, 'max_depth': 3, 'min_child_samples': 75, 'subsample': 0.8446857502222195, 'colsample_bytree': 0.5471265510186669, 'reg_alpha': 1.649979097973048, 'reg_lambda': 0.024238086316683262}


In [ ]:
def xgb_fit(params, X_tr, y_tr, X_val, y_val):
    model = xgb.XGBRegressor(
        n_estimators=5000, random_state=TUNE_SEED, n_jobs=-1, tree_method='hist',
        eval_metric='rmse', early_stopping_rounds=150, **params
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return model.predict(X_val)

def xgb_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
    }
    mean_rmse, std_rmse = cv_rmse_for_params(xgb_fit, params)
    trial.set_user_attr('std', std_rmse)
    return mean_rmse

if HAS_XGB:
    xgb_study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
    xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)
    print("Best XGBoost CV RMSE:", xgb_study.best_value)
    print("Best XGBoost params:", xgb_study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

Best XGBoost CV RMSE: 1142.7522181253157
Best XGBoost params: {'learning_rate': 0.00765983670011748, 'max_depth': 3, 'min_child_weight': 12, 'subsample': 0.5098039632360931, 'colsample_bytree': 0.9443469565455276, 'reg_alpha': 0.02586847036231522, 'reg_lambda': 4.768794641675768, 'gamma': 0.2636885701399665}


In [ ]:
def cat_fit(params, X_tr, y_tr, X_val, y_val):
    model = CatBoostRegressor(
        iterations=5000, random_seed=TUNE_SEED, loss_function='RMSE', verbose=False, **params
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=150, verbose=False)
    return model.predict(X_val)

def cat_objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.0, 5.0),
    }
    mean_rmse, std_rmse = cv_rmse_for_params(cat_fit, params)
    trial.set_user_attr('std', std_rmse)
    return mean_rmse

if HAS_CAT:
    cat_study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
    cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)
    print("Best CatBoost CV RMSE:", cat_study.best_value)
    print("Best CatBoost params:", cat_study.best_params)

In [ ]:
TUNED_LGB = lgb_study.best_params if HAS_LGB else {}
TUNED_XGB = xgb_study.best_params if HAS_XGB else {}
TUNED_CAT = cat_study.best_params if HAS_CAT else {}

def get_models(seed):
    models = {}
    models['hgb'] = HistGradientBoostingRegressor(
        max_iter=700, learning_rate=0.025, max_depth=6, max_leaf_nodes=31,
        l2_regularization=1.0, min_samples_leaf=20,
        early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, random_state=seed
    )
    models['et'] = ExtraTreesRegressor(
        n_estimators=700, max_depth=14, min_samples_leaf=3, max_features=0.7,
        n_jobs=-1, random_state=seed
    )
    models['ridge'] = Ridge(alpha=5.0, random_state=seed)

    if HAS_LGB:
        models['lgb'] = lgb.LGBMRegressor(
            n_estimators=5000, random_state=seed, n_jobs=-1, verbose=-1, **TUNED_LGB
        )
    if HAS_XGB:
        models['xgb'] = xgb.XGBRegressor(
            n_estimators=5000, random_state=seed, n_jobs=-1, tree_method='hist',
            eval_metric='rmse', early_stopping_rounds=150, **TUNED_XGB
        )
    if HAS_CAT:
        models['cat'] = CatBoostRegressor(
            iterations=5000, random_seed=seed, loss_function='RMSE', verbose=False, **TUNED_CAT
        )
    return models

In [ ]:
# Initialize placeholders for OOF predictions and test predictions
oof_preds = np.zeros(len(X))
final_preds = np.zeros(len(X_test))

# Run cross-validation training over multiple seeds (bagging)
for seed in SEEDS:
    print(f"--- Training with Seed {seed} ---")
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        # Instantiate fresh models for this fold
        models = get_models(seed)
        fold_test_preds = []

        # Train each available model and gather predictions
        for name, model in models.items():
            if name == 'xgb':
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            elif name == 'lgb':
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(150, verbose=False)])
            else:
                model.fit(X_tr, y_tr)

            val_pred = model.predict(X_val)
            test_pred = model.predict(X_test)

            # Average model predictions within fold
            oof_preds[val_idx] += val_pred / (len(SEEDS) * len(models))
            final_preds += test_pred / (len(SEEDS) * N_FOLDS * len(models))

# Convert predictions back to original scale (as targets were trained on log1p)
final_preds_original_scale = np.expm1(final_preds)

# Save submission
submission = pd.DataFrame({'id': test_fe['id'], 'total_sales': final_preds_original_scale})
submission = sample_sub[['id']].merge(submission, on='id', how='left')
submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)
print("Saved:", f'{OUTPUT_DIR}/submission.csv')
submission.head()

--- Training with Seed 42 ---
--- Training with Seed 202 ---
--- Training with Seed 777 ---
Saved: /content/drive/MyDrive/DSN_Hackathon/submission.csv


,id,total_sales
0,row_00009,2354.085401
1,row_00015,3824.536248
2,row_00019,2884.417373
3,row_00020,2813.681478
4,row_00023,2290.943513
